# Visualize Diffleop SDF Outputs

This notebook displays generated Diffleop molecules from SDF files in 3D using `py3Dmol`. It is configured to inspect the locally copied RunPod output under `outputs/sampling_dec_001`.


In [ ]:
from pathlib import Path
import sys

def find_diffleop_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "configs").is_dir() and (path / "scripts").is_dir() and (path / "data").is_dir():
            return path
    raise RuntimeError("Could not find the Diffleop root directory")

ROOT = find_diffleop_root()
sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
try:
    import py3Dmol
except ImportError as exc:
    raise ImportError("Install py3Dmol first: mamba run -n diffleop pip install py3Dmol") from exc

from IPython.display import HTML, Markdown, display
from rdkit import Chem
from rdkit.Chem import AllChem
print("py3Dmol, IPython, and RDKit are available")


## Select an Output

The default `RUN_DIR` points to the sampled output copied from RunPod: `outputs/sampling_dec_001`. Completed demo cases are `0` through `4`; `5` is a partial sixth case from the interrupted run.


In [ ]:
RUN_DIR = ROOT / "outputs" / "sampling_dec_001"
OUTPUT_ID = "0"
SAMPLE_INDEX = 0

sample_dir = RUN_DIR / "sdf" / OUTPUT_ID
retain_path = sample_dir / "smiles_retain.smi"

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run output directory was not found: {RUN_DIR}")
if not sample_dir.exists():
    available = sorted(p.name for p in (RUN_DIR / "sdf").iterdir() if p.is_dir())
    raise FileNotFoundError(f"Sample directory was not found: {sample_dir}. Available IDs: {available}")

sdf_candidates = sorted(sample_dir.glob("*.sdf"))
if not sdf_candidates:
    raise FileNotFoundError(f"No SDF files found in {sample_dir}")

sdf_path = sample_dir / f"{SAMPLE_INDEX}.sdf"
if not sdf_path.exists():
    sdf_path = sdf_candidates[0]

available_ids = sorted(p.name for p in (RUN_DIR / "sdf").iterdir() if p.is_dir())
print("run_dir:", RUN_DIR)
print("available_ids:", available_ids)
print("sample_dir:", sample_dir)
print("sdf_path:", sdf_path, sdf_path.exists())
print("retain_path:", retain_path, retain_path.exists())


## Input Metadata

`smiles_retain.smi` records the retained scaffold, masked fragment, original ligand, source ligand SDF, and protein pocket PDB for each demo input.


## Helpers

These helpers collect generated SDF files, resolve optional raw input files, and build fallback reference ligands from the original ligand SMILES when the raw ligand SDF is not available locally.


In [ ]:
METADATA_KEYS = ["retain_smi", "mask_smi", "original_ligand_smi", "ligand_file", "protein_pocket_file"]

def read_retain_metadata(path):
    lines = path.read_text().splitlines()
    return dict(zip(METADATA_KEYS, lines))

def generated_sdf_paths(run_dir=RUN_DIR, include_partial=True):
    paths = sorted((run_dir / "sdf").glob("*/*.sdf"), key=lambda p: (int(p.parent.name), int(p.stem)))
    if include_partial:
        return paths
    return [p for p in paths if p.parent.name in {"0", "1", "2", "3", "4"}]

def resolve_existing_path(rel_path, sample_dir=None):
    rel_path = Path(rel_path)
    candidates = [
        ROOT / "data" / "demo" / "dec" / rel_path,
        ROOT / "data" / rel_path,
        ROOT / rel_path,
    ]
    if sample_dir is not None:
        candidates.append(sample_dir / rel_path.name)
    return next((p for p in candidates if p.exists()), None)

def sdf_text_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    status = AllChem.EmbedMolecule(mol, randomSeed=2024)
    if status == 0:
        AllChem.UFFOptimizeMolecule(mol, maxIters=200)
    mol = Chem.RemoveHs(mol)
    return Chem.MolToMolBlock(mol)

def original_ligand_sdf_text(metadata, sample_dir):
    ligand_path = resolve_existing_path(metadata["ligand_file"], sample_dir)
    if ligand_path is not None:
        return ligand_path.read_text(), ligand_path
    return sdf_text_from_smiles(metadata["original_ligand_smi"]), None

def protein_pdb_text(metadata, sample_dir):
    protein_path = resolve_existing_path(metadata["protein_pocket_file"], sample_dir)
    if protein_path is None:
        return None, None
    return protein_path.read_text(), protein_path

def show_generated(sdf_path, width=700, height=480):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(sdf_path.read_text(), "sdf")
    view.setStyle({"model": 0}, {"stick": {"radius": 0.18, "colorscheme": "greenCarbon"}})
    view.zoomTo()
    return view

def show_comparison(sdf_path, width=820, height=560):
    sample_dir = sdf_path.parent
    metadata = read_retain_metadata(sample_dir / "smiles_retain.smi")
    generated_sdf = sdf_path.read_text()
    original_sdf, ligand_path = original_ligand_sdf_text(metadata, sample_dir)
    pdb, protein_path = protein_pdb_text(metadata, sample_dir)

    view = py3Dmol.view(width=width, height=height)
    model_idx = 0
    if pdb is not None:
        view.addModel(pdb, "pdb")
        view.setStyle({"model": model_idx}, {"cartoon": {"color": "lightgray"}})
        view.addSurface(py3Dmol.VDW, {"opacity": 0.18, "color": "white"}, {"model": model_idx})
        model_idx += 1
    if original_sdf is not None:
        view.addModel(original_sdf, "sdf")
        view.setStyle({"model": model_idx}, {"stick": {"radius": 0.14, "colorscheme": "cyanCarbon"}})
        model_idx += 1
    view.addModel(generated_sdf, "sdf")
    view.setStyle({"model": model_idx}, {"stick": {"radius": 0.20, "colorscheme": "greenCarbon"}})
    view.zoomTo({"model": model_idx})
    return view, metadata, ligand_path, protein_path

all_sdfs = generated_sdf_paths(include_partial=True)
completed_sdfs = generated_sdf_paths(include_partial=False)
print(f"all generated SDF files: {len(all_sdfs)}")
print(f"completed-case SDF files: {len(completed_sdfs)}")
for p in all_sdfs:
    print(p.relative_to(ROOT))


## All Generated Molecules


In [ ]:
for sdf_file in all_sdfs:
    display(Markdown(f"### `{sdf_file.relative_to(ROOT)}`"))
    show_generated(sdf_file).show()


## Compare Generated Molecules with Input Ligand and Pocket

Generated molecules are shown in green. The input ligand is shown in cyan when the raw SDF exists; otherwise the original ligand SMILES is embedded into an approximate 3D conformer and shown in cyan. The protein pocket is added in gray/transparent white only when the raw PDB exists locally.


In [ ]:
for sdf_file in all_sdfs:
    view, metadata, ligand_path, protein_path = show_comparison(sdf_file)
    display(Markdown(f"### `{sdf_file.relative_to(ROOT)}`"))
    print("retain_smi:", metadata["retain_smi"])
    print("mask_smi:", metadata["mask_smi"])
    print("original_ligand_smi:", metadata["original_ligand_smi"])
    print("ligand source:", ligand_path if ligand_path else "SMILES fallback; raw ligand SDF not found")
    print("protein source:", protein_path if protein_path else "raw pocket PDB not found")
    view.show()


## Compare One Selected Case


In [ ]:
selected_view, selected_metadata, selected_ligand_path, selected_protein_path = show_comparison(sdf_path)
print("selected generated SDF:", sdf_path.relative_to(ROOT))
print("ligand source:", selected_ligand_path if selected_ligand_path else "SMILES fallback; raw ligand SDF not found")
print("protein source:", selected_protein_path if selected_protein_path else "raw pocket PDB not found")
selected_view.show()
